# Tool使用的概述

## 1、工具的调用
### 1.1 方式一：直接调用

In [1]:
from langchain_core.tools import tool

@tool
def get_weather(city: str) -> str:
    """
    获取指定城市的天气信息

    参数:
        city: 城市名称，如"北京"、"上海"

    返回:
        天气信息字符串
    """
    # 你的实现
    return city + "晴天，温度 15°C"

In [2]:
get_weather.invoke({'city':'北京'})

'北京晴天，温度 15°C'

### 1.2 方式二：基于模型进行调用

In [2]:
from langchain_community.chat_models import ChatZhipuAI
from dotenv import load_dotenv
import os


# 从.env文件中加载环境变量
load_dotenv(override=True)

ZHIPUAI_API_KEY = os.getenv("ZHIPUAI_API_KEY")
ZHIPUAI_BASE_URL = os.getenv("ZHIPUAI_BASE_URL")

model = ChatZhipuAI(
    model="glm-5.3-flash",
    api_key=ZHIPUAI_API_KEY,
    base_url=ZHIPUAI_BASE_URL
)


python-dotenv could not parse statement starting at line 17


In [ ]:
from langchain_core.tools import tool

# 定义工具
@tool
def get_weather(city: str) -> str:
    """获取指定城市的天气"""
    # 你的实现
    return "晴天，温度 15°C"


# 绑定工具
model_with_tools = model.bind_tools([get_weather])

# AI 可以决定是否调用工具
response = model_with_tools.invoke("北京天气如何？")
# response = model_with_tools.invoke("2 + 3 = ？")

# 检查 AI 是否要调用工具
if response.tool_calls:
    print("AI 想调用工具：", response.tool_calls)
else:
    print("AI 直接回答：", response.content)

AI 想调用工具： [{'name': 'get_weather', 'args': {'city': '北京'}, 'id': 'call_-7180602027993460173', 'type': 'tool_call'}]


# 2、从Message流转看工具调用

In [5]:
from langchain_community.chat_models import ChatZhipuAI
from dotenv import load_dotenv
import os


# 从.env文件中加载环境变量
load_dotenv(override=True)

ZHIPUAI_API_KEY = os.getenv("ZHIPUAI_API_KEY")
ZHIPUAI_BASE_URL = os.getenv("ZHIPUAI_BASE_URL")

model = ChatZhipuAI(
    model="glm-5.3-flash",
    api_key=ZHIPUAI_API_KEY,
    base_url=ZHIPUAI_BASE_URL
)


python-dotenv could not parse statement starting at line 17


不用@tool修饰

In [ ]:
from langchain.messages import HumanMessage, ToolMessage


def get_weather(city: str):
    """获取天气的工具"""
    return f"{city}天气晴朗"


# 将模型和工具绑定
model_with_tools = model.bind_tools([get_weather])
#声明一个消息列表
messages = [
    HumanMessage("今天北京天气如何")
]

# 模型生成调用工具请求
response = model_with_tools.invoke(messages)

# 添加AIMessage到消息列表中
messages.append(response)

tool_calls = response.tool_calls

for tool_call in tool_calls:
    if tool_call["name"] == "get_weather":
        # 拼接出ToolMessage实例
        tool_response = ToolMessage(
            content=get_weather(**tool_call["args"]),
            tool_call_id=tool_call["id"],
            name=tool_call["name"]
        )
        messages.append(tool_response)
print("=====================> messages <=====================")
for msg in messages:
    msg.pretty_print()
print("=====================> messages <=====================")
final_response = model_with_tools.invoke(messages)
print(f"final_response: \n{final_response}")

=====================> messages <=====================
================================ Human Message =================================

今天北京天气如何
================================== Ai Message ==================================
Tool Calls:
  get_weather (call_-7180574230965122317)
 Call ID: call_-7180574230965122317
  Args:
    city: 北京
================================= Tool Message =================================
Name: get_weather

北京天气晴朗
=====================> messages <=====================
final_response: 
content='北京今天天气**晴朗**☀️，是个不错的天气！适合外出活动。' additional_kwargs={} response_metadata={'token_usage': {'completion_tokens': 20, 'completion_tokens_details': {'reasoning_tokens': 0}, 'prompt_tokens': 165, 'prompt_tokens_details': {'cached_tokens': 0}, 'total_tokens': 185}, 'model_name': 'glm-5.3-flash', 'finish_reason': 'stop'} id='lc_run--01a13e9b-1501-7292-b881-f7e95b5d3d6c-0' tool_calls=[] invalid_tool_calls=[]


使用@tool修饰

In [7]:
from langchain.messages import HumanMessage, ToolMessage

@tool
def get_weather(city: str):
    """获取天气的工具"""
    return f"{city}天气晴朗~"


# 将模型和工具绑定
model_with_tools = model.bind_tools([get_weather])

messages = [
    HumanMessage("今天北京天气如何")
]

# 模型生成调用工具请求
response = model_with_tools.invoke(messages)

# 添加AIMessage
messages.append(response)

tool_calls = response.tool_calls

for tool_call in tool_calls:
    if tool_call["name"] == "get_weather":
        # 返回的是ToolMessage类型消息
        tool_response = get_weather.invoke(tool_call)
        print(type(tool_response))
        messages.append(tool_response)

print("=====================> messages <=====================")
for msg in messages:
    msg.pretty_print()
print("=====================> messages <=====================")
final_response = model_with_tools.invoke(messages)
print(f"final_response: \n{final_response}")

<class 'langchain_core.messages.tool.ToolMessage'>
=====================> messages <=====================
================================ Human Message =================================

今天北京天气如何
================================== Ai Message ==================================
Tool Calls:
  get_weather (call_-7180581377790698345)
 Call ID: call_-7180581377790698345
  Args:
    city: 北京
================================= Tool Message =================================
Name: get_weather

北京天气晴朗~
=====================> messages <=====================
final_response: 
content='今天北京天气晴朗☀️，是个不错的好天气！适合外出活动哦～' additional_kwargs={} response_metadata={'token_usage': {'completion_tokens': 66, 'completion_tokens_details': {'reasoning_tokens': 46}, 'prompt_tokens': 166, 'prompt_tokens_details': {'cached_tokens': 0}, 'total_tokens': 232}, 'model_name': 'glm-5.3-flash', 'finish_reason': 'stop'} id='lc_run--01a13ea0-8240-7072-9523-ee270ff0728f-0' tool_calls=[] invalid_tool_calls=[]
